# Notebook 2 — Constraint Testing

This notebook tests **every single function** in `utils/constraints.py` and `utils/data_loader.py`.

Each test builds a minimal, hand-crafted schedule that isolates exactly one behaviour,
so you can see exactly what each function is detecting.

**Run `01_prepare_data.ipynb` first** — this notebook relies on the saved
`outputs/conflict_matrix.pkl` and the three CSV files in `data/`.

---

### Functions tested

| # | Function | File | Type |
|---|----------|------|------|
| 1 | `load_exams` | data_loader | loader |
| 2 | `load_students` | data_loader | loader |
| 3 | `load_rooms` | data_loader | loader |
| 4 | `load_timeslots` | data_loader | loader |
| 5 | `build_conflict_matrix` | data_loader | builder |
| 6 | `load_conflict_matrix` | data_loader | loader |
| 7 | `no_student_clash` | constraints | hard |
| 8 | `no_room_double_booking` | constraints | hard |
| 9 | `room_capacity_ok` | constraints | hard |
| 10 | `one_exam_per_student_per_day` | constraints | hard |
| 11 | `exams_spread_evenly` | constraints | soft |
| 12 | `no_dept_same_day` | constraints | soft |
| 13 | `prefer_morning_slots` | constraints | soft |
| 14 | `count_hard_violations` | constraints | aggregator |
| 15 | `count_soft_violations` | constraints | aggregator |
| 16 | `fitness` | constraints | scorer |
| 17 | `explain_violations` | constraints | reporter |

---
## Step 0 — Imports & Setup

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from utils.data_loader import (
    load_exams,
    load_students,
    load_rooms,
    load_timeslots,
    build_conflict_matrix,
    load_conflict_matrix,
)

from utils.constraints import (
    no_student_clash,
    no_room_double_booking,
    room_capacity_ok,
    one_exam_per_student_per_day,
    exams_spread_evenly,
    no_dept_same_day,
    prefer_morning_slots,
    count_hard_violations,
    count_soft_violations,
    fitness,
    explain_violations,
)

print("✓ All imports successful.")

---
## Part 1 — Data Loader Tests

Tests every function in `utils/data_loader.py` against the real CSV files.

### Test 1 — `load_exams()`

In [ ]:
# ── load_exams ────────────────────────────────────────────────────────────────
# Expected: dict keyed by course code, each value has 'num' and 'students'.

exams = load_exams(path='data/data.csv')

# Basic shape checks
assert isinstance(exams, dict),                          "load_exams should return a dict"
assert len(exams) > 0,                                   "dict must not be empty"

# Every entry must have the correct keys
for course, info in exams.items():
    assert 'num'      in info, f"{course}: missing 'num' key"
    assert 'students' in info, f"{course}: missing 'students' key"
    assert isinstance(info['num'],      int),  f"{course}: 'num' must be int"
    assert isinstance(info['students'], list), f"{course}: 'students' must be list"
    assert info['num'] == len(info['students']), (
        f"{course}: 'num' ({info['num']}) does not match "
        f"len(students) ({len(info['students'])})"
    )

# Spot-check a known exam from the real data
assert 'ACCT201' in exams,            "ACCT201 should be in the dataset"
assert exams['ACCT201']['num'] == 205, "ACCT201 should have 205 students"

print(f"✓ load_exams  |  {len(exams)} exams loaded")
print(f"  Largest exam : {max(exams, key=lambda k: exams[k]['num'])}  "
      f"({max(e['num'] for e in exams.values())} students)")
print(f"  Smallest exam: {min(exams, key=lambda k: exams[k]['num'])}  "
      f"({min(e['num'] for e in exams.values())} students)")

### Test 2 — `load_students()`

In [ ]:
# ── load_students ─────────────────────────────────────────────────────────────
# Expected: dict keyed by student ID, each value is a list of course codes.

students = load_students(path='data/IDs.csv')

assert isinstance(students, dict), "load_students should return a dict"
assert len(students) > 0,          "dict must not be empty"

# Every entry must be a non-empty list of strings
for sid, courses in students.items():
    assert isinstance(courses, list),           f"{sid}: value must be a list"
    assert len(courses) > 0,                    f"{sid}: student has no courses"
    assert all(isinstance(c, str) for c in courses), \
                                                f"{sid}: courses must be strings"

# Cross-check: every course a student claims should exist in exams
all_exams = set(exams.keys())
unknown = [
    (sid, c)
    for sid, courses in students.items()
    for c in courses
    if c not in all_exams
]
assert len(unknown) == 0, f"Students reference unknown exams: {unknown[:5]}"

# Distribution of courses per student
course_counts = [len(v) for v in students.values()]
print(f"✓ load_students  |  {len(students)} students loaded")
print(f"  Min courses/student : {min(course_counts)}")
print(f"  Max courses/student : {max(course_counts)}")
print(f"  Avg courses/student : {sum(course_counts)/len(course_counts):.2f}")

### Test 3 — `load_rooms()`

In [ ]:
# ── load_rooms ────────────────────────────────────────────────────────────────
# Expected: list of dicts with keys room_id, building, capacity.

rooms = load_rooms(path='data/rooms.csv')

assert isinstance(rooms, list), "load_rooms should return a list"
assert len(rooms) > 0,          "must have at least one usable room"

for r in rooms:
    assert 'room_id'  in r, f"missing 'room_id' in {r}"
    assert 'building' in r, f"missing 'building' in {r}"
    assert 'capacity' in r, f"missing 'capacity' in {r}"
    assert isinstance(r['capacity'], int), f"capacity must be int in {r}"
    assert r['capacity'] > 0,             f"capacity must be > 0 in {r}"

# Room IDs must be unique
ids = [r['room_id'] for r in rooms]
assert len(ids) == len(set(ids)), "room_id values must be unique"

total_seats = sum(r['capacity'] for r in rooms)
print(f"✓ load_rooms  |  {len(rooms)} usable rooms loaded")
print(f"  Total seats     : {total_seats}")
print(f"  Largest room    : {max(rooms, key=lambda r: r['capacity'])['room_id']}  "
      f"({max(r['capacity'] for r in rooms)} seats)")
print(f"  Smallest room   : {min(rooms, key=lambda r: r['capacity'])['room_id']}  "
      f"({min(r['capacity'] for r in rooms)} seats)")

### Test 4 — `load_timeslots()`

In [ ]:
# ── load_timeslots ────────────────────────────────────────────────────────────
# Expected: 60 slots (15 working days × 4 slots/day).
# Friday must be excluded. Slots run 08-10, 10-12, 12-14, 14-16.

timeslots = load_timeslots()

assert isinstance(timeslots, list), "load_timeslots should return a list"

# Every entry must have the correct keys
for ts in timeslots:
    for key in ('slot_id', 'date', 'day', 'time'):
        assert key in ts, f"timeslot missing key '{key}': {ts}"

# slot_ids must be 0..N-1 with no gaps
slot_ids = [ts['slot_id'] for ts in timeslots]
assert slot_ids == list(range(len(timeslots))), "slot_ids must be sequential from 0"

# No Fridays allowed
fridays = [ts for ts in timeslots if ts['day'] == 'Friday']
assert len(fridays) == 0, f"Found {len(fridays)} Friday slots — Fridays should be excluded"

# Exactly 4 time blocks per day
valid_times = {'08:00-10:00', '10:00-12:00', '12:00-14:00', '14:00-16:00'}
for ts in timeslots:
    assert ts['time'] in valid_times, f"Unexpected time block: {ts['time']}"

# Count working days
unique_dates = sorted(set(ts['date'] for ts in timeslots))
assert len(timeslots) == len(unique_dates) * 4, \
    f"Expected 4 slots per day, got {len(timeslots)} total for {len(unique_dates)} days"

# Date range: 31 May to 16 June 2025
assert unique_dates[0]  == '2025-05-31', f"First date should be 2025-05-31, got {unique_dates[0]}"
assert unique_dates[-1] == '2025-06-16', f"Last date should be 2025-06-16, got {unique_dates[-1]}"

print(f"✓ load_timeslots  |  {len(timeslots)} slots across {len(unique_dates)} working days")
print(f"  Date range : {unique_dates[0]}  →  {unique_dates[-1]}")
print(f"  Time blocks: {sorted(valid_times)}")

### Test 5 — `build_conflict_matrix()`

In [ ]:
# ── build_conflict_matrix ─────────────────────────────────────────────────────
# Builds a tiny 3-exam matrix from scratch with known student overlap,
# then verifies the correct pairs are flagged.
#
# Setup:
#   EXAM_A: students {s1, s2}
#   EXAM_B: students {s2, s3}   → shares s2 with A  ⟹ A-B conflict
#   EXAM_C: students {s4, s5}   → shares nobody     ⟹ no conflicts

mini_exams = {
    'EXAM_A': {'num': 2, 'students': ['s1', 's2']},
    'EXAM_B': {'num': 2, 'students': ['s2', 's3']},
    'EXAM_C': {'num': 2, 'students': ['s4', 's5']},
}

import tempfile, os
with tempfile.TemporaryDirectory() as tmp:
    save_path = os.path.join(tmp, 'mini_cm.pkl')
    mini_cm = build_conflict_matrix(mini_exams, save_path=save_path)

# A and B share s2 → they conflict with each other
assert 'EXAM_B' in mini_cm['EXAM_A'], "EXAM_A should conflict with EXAM_B (shared student s2)"
assert 'EXAM_A' in mini_cm['EXAM_B'], "Conflict must be symmetric: B→A"

# C shares nobody
assert 'EXAM_C' not in mini_cm['EXAM_A'], "EXAM_A should NOT conflict with EXAM_C"
assert 'EXAM_C' not in mini_cm['EXAM_B'], "EXAM_B should NOT conflict with EXAM_C"
assert len(mini_cm['EXAM_C']) == 0,       "EXAM_C should have zero conflicts"

print("✓ build_conflict_matrix  |  mini test passed")
print(f"  EXAM_A conflicts: {mini_cm['EXAM_A']}")
print(f"  EXAM_B conflicts: {mini_cm['EXAM_B']}")
print(f"  EXAM_C conflicts: {mini_cm['EXAM_C']}")

### Test 6 — `load_conflict_matrix()`

In [ ]:
# ── load_conflict_matrix ──────────────────────────────────────────────────────
# Loads the pre-built matrix from disk and verifies its structure.

conflict_matrix = load_conflict_matrix(path='outputs/conflict_matrix.pkl')

assert isinstance(conflict_matrix, dict), "conflict_matrix should be a dict"
assert len(conflict_matrix) == len(exams), (
    f"conflict_matrix has {len(conflict_matrix)} entries but exams has {len(exams)}"
)

# Keys must match exam codes exactly
assert set(conflict_matrix.keys()) == set(exams.keys()), \
    "conflict_matrix keys must match exam codes"

# Every value must be a set of strings
for exam, conflicts in conflict_matrix.items():
    assert isinstance(conflicts, set), f"{exam}: conflicts must be a set"
    for c in conflicts:
        assert isinstance(c, str), f"{exam}: conflict entries must be strings"
        assert c in exams,         f"{exam}: conflicting exam '{c}' not in exam list"

# Matrix must be symmetric: if A conflicts B, then B conflicts A
for exam, conflicts in conflict_matrix.items():
    for other in conflicts:
        assert exam in conflict_matrix[other], \
            f"Asymmetry: {exam}→{other} exists but {other}→{exam} does not"

total_conflict_pairs = sum(len(v) for v in conflict_matrix.values()) // 2
print(f"✓ load_conflict_matrix  |  loaded successfully")
print(f"  Exams in matrix        : {len(conflict_matrix)}")
print(f"  Total conflicting pairs: {total_conflict_pairs}")

---
## Part 2 — Constraint Function Tests

Each test below builds a **hand-crafted mini schedule** that either:
- **Triggers exactly N violations** of the function under test, or
- **Passes with 0 violations** (clean schedule check).

Mini data shared across all constraint tests:

In [ ]:
# ── Shared mini test data ─────────────────────────────────────────────────────
# A tiny, fully controlled dataset used by all constraint tests below.
# Real data is also tested at the end of each section.

mini_exams = {
    'CS101': {'num': 2, 'students': ['s1', 's2']},
    'CS102': {'num': 2, 'students': ['s2', 's3']},   # shares s2 with CS101
    'MA101': {'num': 2, 'students': ['s4', 's5']},   # no overlap
    'MA102': {'num': 1, 'students': ['s6']},          # tiny exam
    'PH101': {'num': 3, 'students': ['s1', 's7', 's8']},  # shares s1 with CS101
}

mini_rooms = [
    {'room_id': 'R1', 'building': 'Block A', 'capacity': 30},
    {'room_id': 'R2', 'building': 'Block B', 'capacity': 5},
]

# 3 days × 4 slots = 12 slots; Day 1 = 2025-05-31 (Saturday)
mini_timeslots = [
    # Day 1 — 2025-05-31
    {'slot_id': 0,  'date': '2025-05-31', 'day': 'Saturday',  'time': '08:00-10:00'},
    {'slot_id': 1,  'date': '2025-05-31', 'day': 'Saturday',  'time': '10:00-12:00'},
    {'slot_id': 2,  'date': '2025-05-31', 'day': 'Saturday',  'time': '12:00-14:00'},
    {'slot_id': 3,  'date': '2025-05-31', 'day': 'Saturday',  'time': '14:00-16:00'},
    # Day 2 — 2025-06-01
    {'slot_id': 4,  'date': '2025-06-01', 'day': 'Sunday',    'time': '08:00-10:00'},
    {'slot_id': 5,  'date': '2025-06-01', 'day': 'Sunday',    'time': '10:00-12:00'},
    {'slot_id': 6,  'date': '2025-06-01', 'day': 'Sunday',    'time': '12:00-14:00'},
    {'slot_id': 7,  'date': '2025-06-01', 'day': 'Sunday',    'time': '14:00-16:00'},
    # Day 3 — 2025-06-02
    {'slot_id': 8,  'date': '2025-06-02', 'day': 'Monday',    'time': '08:00-10:00'},
    {'slot_id': 9,  'date': '2025-06-02', 'day': 'Monday',    'time': '10:00-12:00'},
    {'slot_id': 10, 'date': '2025-06-02', 'day': 'Monday',    'time': '12:00-14:00'},
    {'slot_id': 11, 'date': '2025-06-02', 'day': 'Monday',    'time': '14:00-16:00'},
]

# Conflict matrix for mini_exams
mini_cm = {
    'CS101': {'CS102', 'PH101'},  # shares s2 with CS102, shares s1 with PH101
    'CS102': {'CS101'},
    'MA101': set(),
    'MA102': set(),
    'PH101': {'CS101'},
}

print("✓ Mini test data ready.")
print(f"  {len(mini_exams)} exams, {len(mini_rooms)} rooms, {len(mini_timeslots)} slots")

---
### Test 7 — `no_student_clash()`

No student may sit two exams in the same timeslot.

In [ ]:
# ── no_student_clash ──────────────────────────────────────────────────────────

# --- Case A: 0 violations ---
# CS101 and CS102 conflict but are in DIFFERENT slots.
schedule_clean = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 1},  # different slot
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 2},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 3},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 4},
]
result_clean = no_student_clash(schedule_clean, mini_cm)
assert result_clean == 0, f"Expected 0 violations, got {result_clean}"

# --- Case B: 1 violation ---
# CS101 and CS102 are in the SAME slot → 1 clash (they share s2).
schedule_one_clash = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 0},  # same slot!
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 2},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 3},
]
result_one = no_student_clash(schedule_one_clash, mini_cm)
assert result_one == 1, f"Expected 1 violation, got {result_one}"

# --- Case C: 3 violations ---
# CS101, CS102, and PH101 all in slot 0.
# Pairs: (CS101,CS102) → clash, (CS101,PH101) → clash, (CS102,PH101) → no shared student.
# So 2 violations.
schedule_two_clashes = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 0},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 0},  # also slot 0
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 2},
]
result_two = no_student_clash(schedule_two_clashes, mini_cm)
assert result_two == 2, f"Expected 2 violations (CS101-CS102 and CS101-PH101), got {result_two}"

# --- Real data sanity check ---
# Build a deliberately bad schedule: put every exam in slot 0.
all_in_slot0 = [{'exam': e, 'room_id': rooms[0]['room_id'], 'slot_id': 0}
                for e in list(exams.keys())[:20]]  # first 20 exams only
real_clashes = no_student_clash(all_in_slot0, conflict_matrix)
assert real_clashes > 0, "Jamming 20 exams into slot 0 should produce student clashes"

print("✓ no_student_clash")
print(f"  Case A (clean)         : {result_clean} violations  (expected 0)")
print(f"  Case B (1 clash)       : {result_one}  violation   (expected 1)")
print(f"  Case C (2 clashes)     : {result_two}  violations  (expected 2)")
print(f"  Real data (20@slot0)   : {real_clashes} violations")

### Test 8 — `no_room_double_booking()`

No two exams may use the same room at the same time.

In [ ]:
# ── no_room_double_booking ────────────────────────────────────────────────────

# --- Case A: 0 violations ---
# Each (room, slot) pair is unique.
schedule_no_double = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 0},  # different room, same slot → OK
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},  # same room, different slot → OK
]
result_clean = no_room_double_booking(schedule_no_double)
assert result_clean == 0, f"Expected 0, got {result_clean}"

# --- Case B: 1 violation ---
# CS101 and MA101 both in R1 at slot 0.
schedule_one_double = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 0},  # same room AND slot!
    {'exam': 'MA102', 'room_id': 'R2', 'slot_id': 1},
]
result_one = no_room_double_booking(schedule_one_double)
assert result_one == 1, f"Expected 1, got {result_one}"

# --- Case C: 2 violations ---
# (R1, slot0) used by 3 exams → 2 extra bookings = 2 violations.
schedule_two_doubles = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'MA102', 'room_id': 'R2', 'slot_id': 1},
]
result_two = no_room_double_booking(schedule_two_doubles)
assert result_two == 2, f"Expected 2, got {result_two}"

# --- Case D: all exams spread across unique (room, slot) pairs → 0 violations ---
schedule_all_unique = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 1},
    {'exam': 'MA101', 'room_id': 'R2', 'slot_id': 0},
    {'exam': 'MA102', 'room_id': 'R2', 'slot_id': 1},
]
result_unique = no_room_double_booking(schedule_all_unique)
assert result_unique == 0, f"Expected 0, got {result_unique}"

print("✓ no_room_double_booking")
print(f"  Case A (no doubles)    : {result_clean}  (expected 0)")
print(f"  Case B (1 double)      : {result_one}  (expected 1)")
print(f"  Case C (2 doubles)     : {result_two}  (expected 2)")
print(f"  Case D (all unique)    : {result_unique}  (expected 0)")

### Test 9 — `room_capacity_ok()`

The room assigned to an exam must have enough seats for all enrolled students.

In [ ]:
# ── room_capacity_ok ──────────────────────────────────────────────────────────
# mini_exams: CS101=2 students, MA101=2, MA102=1, PH101=3
# mini_rooms: R1=30 seats, R2=5 seats

# --- Case A: 0 violations — every exam fits in its room ---
schedule_fits = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # 2 students → 30 seats ✓
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},  # 2 students → 30 seats ✓
    {'exam': 'PH101', 'room_id': 'R2', 'slot_id': 2},  # 3 students → 5 seats  ✓
]
result_clean = room_capacity_ok(schedule_fits, mini_exams, mini_rooms)
assert result_clean == 0, f"Expected 0, got {result_clean}"

# --- Case B: 1 violation — build a room that's too small ---
# Make a tiny room with capacity 1, assign CS101 (2 students) to it.
tiny_rooms = [{'room_id': 'TINY', 'building': 'X', 'capacity': 1}]
schedule_overflow = [
    {'exam': 'CS101', 'room_id': 'TINY', 'slot_id': 0},  # 2 > 1 → violation
    {'exam': 'MA102', 'room_id': 'TINY', 'slot_id': 1},  # 1 = 1 → OK
]
result_one = room_capacity_ok(schedule_overflow, mini_exams, tiny_rooms)
assert result_one == 1, f"Expected 1, got {result_one}"

# --- Case C: multiple violations ---
schedule_multi_overflow = [
    {'exam': 'CS101', 'room_id': 'TINY', 'slot_id': 0},  # 2 > 1 → violation
    {'exam': 'PH101', 'room_id': 'TINY', 'slot_id': 1},  # 3 > 1 → violation
    {'exam': 'MA102', 'room_id': 'TINY', 'slot_id': 2},  # 1 = 1 → OK
]
result_two = room_capacity_ok(schedule_multi_overflow, mini_exams, tiny_rooms)
assert result_two == 2, f"Expected 2, got {result_two}"

# --- Real data: count how many real exams overflow rooms[0] (smallest plausible room) ---
smallest_room = min(rooms, key=lambda r: r['capacity'])
real_overflow_schedule = [
    {'exam': e, 'room_id': smallest_room['room_id'], 'slot_id': i}
    for i, e in enumerate(list(exams.keys())[:10])
]
real_overflows = room_capacity_ok(real_overflow_schedule, exams, rooms)

print("✓ room_capacity_ok")
print(f"  Case A (all fit)       : {result_clean}  (expected 0)")
print(f"  Case B (1 overflow)    : {result_one}  (expected 1)")
print(f"  Case C (2 overflows)   : {result_two}  (expected 2)")
print(f"  Real data (smallest rm): {real_overflows} overflows out of 10 exams")

### Test 10 — `one_exam_per_student_per_day()`

A student may not sit two exams on the same calendar day (even in different timeslots).

In [ ]:
# ── one_exam_per_student_per_day ──────────────────────────────────────────────
# mini_timeslots: slots 0-3 are Day1 (2025-05-31), slots 4-7 are Day2.
# CS101 ↔ CS102 conflict (share s2). CS101 ↔ PH101 conflict (share s1).

# --- Case A: 0 violations — conflicting exams on DIFFERENT days ---
schedule_diff_days = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # Day 1, slot 0
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 4},  # Day 2, slot 4
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 8},  # Day 3
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 2},
]
result_clean = one_exam_per_student_per_day(schedule_diff_days, mini_cm, mini_timeslots)
assert result_clean == 0, f"Expected 0, got {result_clean}"

# --- Case B: 1 violation — CS101 & CS102 on same day (different slots) ---
# Slot 0 and slot 1 are both on Day 1 (2025-05-31).
schedule_same_day = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # Day 1, slot 0
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 1},  # Day 1, slot 1  ← same day!
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 4},  # Day 2
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 5},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 6},
]
result_one = one_exam_per_student_per_day(schedule_same_day, mini_cm, mini_timeslots)
assert result_one == 1, f"Expected 1, got {result_one}"

# --- Case C: 2 violations — CS101, CS102, and PH101 all on Day 1 ---
# Pairs on Day 1: (CS101,CS102) → clash, (CS101,PH101) → clash, (CS102,PH101) → no clash.
schedule_day1_triple = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 1},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 2},  # Day 1, slot 2 ← also Day 1!
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 4},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},
]
result_two = one_exam_per_student_per_day(schedule_day1_triple, mini_cm, mini_timeslots)
assert result_two == 2, f"Expected 2, got {result_two}"

print("✓ one_exam_per_student_per_day")
print(f"  Case A (diff days)     : {result_clean}  (expected 0)")
print(f"  Case B (1 same-day)    : {result_one}  (expected 1)")
print(f"  Case C (2 same-day)    : {result_two}  (expected 2)")

---
### Test 11 — `exams_spread_evenly()`

Exams should not all cluster on the same day — days more than 2 above the daily average are penalised.

In [ ]:
# ── exams_spread_evenly ───────────────────────────────────────────────────────
# mini_timeslots: 3 days, 4 slots/day.
# With 5 exams and 3 days: ideal = 5/3 ≈ 1.67 per day.
# Threshold for violation: > 1.67 + 2 = 3.67 → need 4+ exams on a day to trigger.

# --- Case A: 0 violations — evenly spread ---
# Day1: 2 exams, Day2: 2 exams, Day3: 1 exam → max = 2, well below threshold.
schedule_even = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # Day1
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 1},  # Day1
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 4},  # Day2
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},  # Day2
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 8},  # Day3
]
result_clean = exams_spread_evenly(schedule_even, mini_timeslots)
assert result_clean == 0, f"Expected 0, got {result_clean}"

# --- Case B: 1 violation — 4 exams crammed into Day 1 ---
# 5 exams total, 3 days → ideal ≈ 1.67; threshold = 3.67, so 4 > 3.67 triggers it.
schedule_clustered = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # Day1
    {'exam': 'CS102', 'room_id': 'R2', 'slot_id': 1},  # Day1
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 2},  # Day1
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 3},  # Day1  ← 4 on one day!
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 4},  # Day2
]
result_cluster = exams_spread_evenly(schedule_clustered, mini_timeslots)
assert result_cluster == 1, f"Expected 1 (Day1 overloaded), got {result_cluster}"

# --- Real data sanity: a round-robin spread across all real timeslots → 0 violations ---
real_spread_schedule = [
    {'exam': e, 'room_id': rooms[0]['room_id'], 'slot_id': i % len(timeslots)}
    for i, e in enumerate(exams.keys())
]
real_spread_result = exams_spread_evenly(real_spread_schedule, timeslots)

print("✓ exams_spread_evenly")
print(f"  Case A (even spread)   : {result_clean}  (expected 0)")
print(f"  Case B (Day1 overload) : {result_cluster}  (expected 1)")
print(f"  Real data (round-robin): {real_spread_result} overloaded days")

### Test 12 — `no_dept_same_day()`

More than 3 exams from the same department on one day triggers a penalty.

In [ ]:
# ── no_dept_same_day ──────────────────────────────────────────────────────────
# Department is derived from the alphabetic prefix of the exam code.
# CS101, CS102 → dept 'CS'.  MA101, MA102 → dept 'MA'.  PH101 → dept 'PH'.

# --- Case A: 0 violations — each dept has at most 2 exams per day ---
schedule_no_dept_issue = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # Day1, CS
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 1},  # Day1, CS  → 2 CS on Day1, OK
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 4},  # Day2, MA
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},  # Day2, MA  → 2 MA on Day2, OK
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 8},  # Day3, PH
]
result_clean = no_dept_same_day(schedule_no_dept_issue, mini_timeslots)
assert result_clean == 0, f"Expected 0, got {result_clean}"

# --- Case B: 1 violation — 4 CS exams on Day 1 (1 above the threshold of 3) ---
# Uses a richer mini exam set for this test only
dept_timeslots = [
    {'slot_id': 0, 'date': '2025-05-31', 'day': 'Saturday', 'time': '08:00-10:00'},
    {'slot_id': 1, 'date': '2025-05-31', 'day': 'Saturday', 'time': '10:00-12:00'},
    {'slot_id': 2, 'date': '2025-05-31', 'day': 'Saturday', 'time': '12:00-14:00'},
    {'slot_id': 3, 'date': '2025-05-31', 'day': 'Saturday', 'time': '14:00-16:00'},
    {'slot_id': 4, 'date': '2025-06-01', 'day': 'Sunday',   'time': '08:00-10:00'},
]
schedule_dept_overload = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # Day1
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 1},  # Day1
    {'exam': 'CS201', 'room_id': 'R1', 'slot_id': 2},  # Day1
    {'exam': 'CS202', 'room_id': 'R1', 'slot_id': 3},  # Day1  ← 4 CS on one day!
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 4},  # Day2, different dept
]
result_dept = no_dept_same_day(schedule_dept_overload, dept_timeslots)
# 4 CS on one day → 4-3 = 1 violation
assert result_dept == 1, f"Expected 1 violation (4 CS on Day1), got {result_dept}"

# --- Case C: 2 violations — 5 CS exams on Day 1 (5-3 = 2 extra) ---
dept_timeslots_5slots = dept_timeslots + [
    {'slot_id': 5, 'date': '2025-05-31', 'day': 'Saturday', 'time': '08:00-10:00'},
]
schedule_five_cs = schedule_dept_overload + [
    {'exam': 'CS301', 'room_id': 'R1', 'slot_id': 5},  # Day1, 5th CS exam!
]
result_five = no_dept_same_day(schedule_five_cs, dept_timeslots_5slots)
assert result_five == 2, f"Expected 2 violations (5-3=2), got {result_five}"

print("✓ no_dept_same_day")
print(f"  Case A (no dept issue) : {result_clean}  (expected 0)")
print(f"  Case B (4 CS on Day1)  : {result_dept}  (expected 1)")
print(f"  Case C (5 CS on Day1)  : {result_five}  (expected 2)")

### Test 13 — `prefer_morning_slots()`

Each exam in an afternoon slot (12:00-14:00 or 14:00-16:00) adds 1 penalty.

In [ ]:
# ── prefer_morning_slots ──────────────────────────────────────────────────────
# Morning slots: 08:00-10:00, 10:00-12:00
# Afternoon slots: 12:00-14:00, 14:00-16:00  (each adds 1 violation)

# Recall mini_timeslots:
#   slot 0 = 08:00-10:00  (morning)     slot 2 = 12:00-14:00  (afternoon)
#   slot 1 = 10:00-12:00  (morning)     slot 3 = 14:00-16:00  (afternoon)

# --- Case A: 0 violations — all exams in morning slots ---
schedule_morning = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # 08:00-10:00
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 1},  # 10:00-12:00
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 4},  # 08:00-10:00 (Day2)
]
result_morning = prefer_morning_slots(schedule_morning, mini_timeslots)
assert result_morning == 0, f"Expected 0, got {result_morning}"

# --- Case B: 2 violations — 2 exams in afternoon slots ---
schedule_mixed = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},  # morning
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 2},  # 12:00-14:00 → violation
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 3},  # 14:00-16:00 → violation
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 1},  # morning
]
result_two = prefer_morning_slots(schedule_mixed, mini_timeslots)
assert result_two == 2, f"Expected 2, got {result_two}"

# --- Case C: all exams in afternoon slots ---
schedule_all_afternoon = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 2},  # 12:00-14:00
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 3},  # 14:00-16:00
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 6},  # 12:00-14:00 (Day2)
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 7},  # 14:00-16:00 (Day2)
]
result_all_pm = prefer_morning_slots(schedule_all_afternoon, mini_timeslots)
assert result_all_pm == 4, f"Expected 4 (all afternoon), got {result_all_pm}"

# --- Real data: count afternoon slots in a round-robin real schedule ---
real_schedule = [
    {'exam': e, 'room_id': rooms[0]['room_id'], 'slot_id': i % len(timeslots)}
    for i, e in enumerate(exams.keys())
]
real_pm = prefer_morning_slots(real_schedule, timeslots)
# Expect roughly half of all exams to land in afternoon slots
assert 0 < real_pm < len(exams), "Real data should have some (but not all) afternoon exams"

print("✓ prefer_morning_slots")
print(f"  Case A (all morning)   : {result_morning}  (expected 0)")
print(f"  Case B (2 afternoon)   : {result_two}  (expected 2)")
print(f"  Case C (all afternoon) : {result_all_pm}  (expected 4)")
print(f"  Real data (round-robin): {real_pm} afternoon exams out of {len(exams)}")

---
## Part 3 — Aggregator Tests

Tests for the functions that combine individual constraint results.

### Test 14 — `count_hard_violations()`

In [ ]:
# ── count_hard_violations ─────────────────────────────────────────────────────
# Must equal the sum of the four individual hard constraint functions.

# Build a schedule with deliberate hard violations:
#   - CS101 & CS102 in slot 0, room R1  → room double-booking (1) + student clash (1)
#   - CS101 & CS102 both in slot 0      → one_exam_per_student_per_day clash on Day1 (1)
#   - PH101 in tiny room (capacity 1)   → capacity violation (1)
tiny_rm = [{'room_id': 'TINY', 'building': 'X', 'capacity': 1}]

test_schedule = [
    {'exam': 'CS101', 'room_id': 'R1',  'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1',  'slot_id': 0},   # double-booking R1 @ slot0; shared student s2
    {'exam': 'MA101', 'room_id': 'R2',  'slot_id': 4},
    {'exam': 'MA102', 'room_id': 'R2',  'slot_id': 5},
    {'exam': 'PH101', 'room_id': 'TINY','slot_id': 6},   # PH101 has 3 students, TINY has 1 → overflow
]

mixed_rooms = mini_rooms + tiny_rm

h1 = no_student_clash(test_schedule, mini_cm)
h2 = no_room_double_booking(test_schedule)
h3 = room_capacity_ok(test_schedule, mini_exams, mixed_rooms)
h4 = one_exam_per_student_per_day(test_schedule, mini_cm, mini_timeslots)
expected_total = h1 + h2 + h3 + h4

total = count_hard_violations(test_schedule, mini_cm, mini_exams, mixed_rooms, mini_timeslots)

assert total == expected_total, (
    f"count_hard_violations={total} but h1+h2+h3+h4={expected_total}"
)
assert total > 0, "Test schedule should have hard violations"

# A perfectly clean schedule should give 0
perfect_schedule = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 4},  # different day from CS101
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 8},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 9},
]
zero_hard = count_hard_violations(perfect_schedule, mini_cm, mini_exams, mini_rooms, mini_timeslots)
assert zero_hard == 0, f"Perfect schedule should have 0 hard violations, got {zero_hard}"

print("✓ count_hard_violations")
print(f"  Individual: h1={h1}, h2={h2}, h3={h3}, h4={h4}  → sum={expected_total}")
print(f"  Aggregated : {total}  (matches)")
print(f"  Perfect schedule: {zero_hard}  (expected 0)")

### Test 15 — `count_soft_violations()`

In [ ]:
# ── count_soft_violations ─────────────────────────────────────────────────────
# Must equal the sum of the three individual soft constraint functions.

# Use the clustered schedule from Test 11 (Day1 overloaded + all in afternoon)
soft_test_schedule = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 2},  # Day1, afternoon
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 3},  # Day1, afternoon
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 0},  # Day1, morning
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 1},  # Day1, morning
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 4},  # Day2, morning
]

s1 = exams_spread_evenly(soft_test_schedule, mini_timeslots)
s2 = no_dept_same_day(soft_test_schedule, mini_timeslots)
s3 = prefer_morning_slots(soft_test_schedule, mini_timeslots)
expected_soft = s1 + s2 + s3

total_soft = count_soft_violations(soft_test_schedule, mini_timeslots)

assert total_soft == expected_soft, (
    f"count_soft_violations={total_soft} but s1+s2+s3={expected_soft}"
)

print("✓ count_soft_violations")
print(f"  Individual: s1={s1}, s2={s2}, s3={s3}  → sum={expected_soft}")
print(f"  Aggregated : {total_soft}  (matches)")

---
## Part 4 — Fitness & Explain Tests

### Test 16 — `fitness()`

Score = -(hard × 100 + soft × 1). A perfect schedule scores 0; any violation makes it negative.

In [ ]:
# ── fitness ───────────────────────────────────────────────────────────────────

# --- Case A: perfect mini schedule → fitness should be 0 ---
# (All hard constraints satisfied; we accept whatever soft violations emerge.)
perfect_mini = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 4},
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 8},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 9},
]
hard_perfect = count_hard_violations(perfect_mini, mini_cm, mini_exams, mini_rooms, mini_timeslots)
soft_perfect = count_soft_violations(perfect_mini, mini_timeslots)
expected_perfect_score = -(hard_perfect * 100 + soft_perfect)

score_perfect = fitness(perfect_mini, mini_cm, mini_exams, mini_rooms, mini_timeslots)
assert score_perfect == expected_perfect_score, \
    f"Expected {expected_perfect_score}, got {score_perfect}"

# --- Case B: formula verification — force known violations ---
# Same room+slot for CS101 & CS102 → 1 room double-booking + 1 student clash + 1 same-day
# = 3 hard violations. Plus whatever soft falls out.
bad_mini = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 0},  # clash!
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 4},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 9},
]
hard_bad = count_hard_violations(bad_mini, mini_cm, mini_exams, mini_rooms, mini_timeslots)
soft_bad = count_soft_violations(bad_mini, mini_timeslots)
expected_bad_score = -(hard_bad * 100 + soft_bad)

score_bad = fitness(bad_mini, mini_cm, mini_exams, mini_rooms, mini_timeslots)
assert score_bad == expected_bad_score, \
    f"Expected {expected_bad_score}, got {score_bad}"
assert score_bad < score_perfect, \
    "Bad schedule must score worse (more negative) than perfect"

# --- Case C: hard violations dominate soft violations ---
# 1 hard violation (penalty 100) must outweigh any reasonable number of soft violations.
assert hard_bad * 100 > soft_perfect, \
    "Hard penalty (100/violation) must dominate soft penalties (1/violation)"

# --- Real data: fitness of real conflict_matrix round-robin schedule ---
real_schedule = [
    {'exam': e, 'room_id': rooms[0]['room_id'], 'slot_id': i % len(timeslots)}
    for i, e in enumerate(exams.keys())
]
real_score = fitness(real_schedule, conflict_matrix, exams, rooms, timeslots)
assert isinstance(real_score, int), "fitness should return an int"
assert real_score <= 0,             "fitness score must be <= 0"

print("✓ fitness")
print(f"  Perfect mini score     : {score_perfect}  (hard={hard_perfect}, soft={soft_perfect})")
print(f"  Bad mini score         : {score_bad}  (hard={hard_bad}, soft={soft_bad})")
print(f"  Real data score        : {real_score}")

### Test 17 — `explain_violations()`

Produces a printed violation report — tested for output correctness, not crashes.

In [ ]:
# ── explain_violations ────────────────────────────────────────────────────────
# This function prints a report; we verify it runs without error and that the
# numbers it uses are consistent with the individual constraint functions.

import io, contextlib

test_mini_schedule = [
    {'exam': 'CS101', 'room_id': 'R1', 'slot_id': 0},
    {'exam': 'CS102', 'room_id': 'R1', 'slot_id': 0},  # double-booking + student clash + same-day
    {'exam': 'MA101', 'room_id': 'R1', 'slot_id': 4},
    {'exam': 'MA102', 'room_id': 'R1', 'slot_id': 5},
    {'exam': 'PH101', 'room_id': 'R1', 'slot_id': 9},
]

# Capture stdout to verify output content
output_buffer = io.StringIO()
with contextlib.redirect_stdout(output_buffer):
    explain_violations(test_mini_schedule, mini_cm, mini_exams, mini_rooms, mini_timeslots)

report = output_buffer.getvalue()

# Verify essential sections are present
assert 'HARD CONSTRAINTS' in report,   "Report must contain HARD CONSTRAINTS section"
assert 'SOFT CONSTRAINTS' in report,   "Report must contain SOFT CONSTRAINTS section"
assert 'FITNESS SCORE'    in report,   "Report must contain FITNESS SCORE line"
assert 'SCHEDULE VIOLATION REPORT' in report, "Report must have a title"

# Verify numbers in the report match what the individual functions return
h1 = no_student_clash(test_mini_schedule, mini_cm)
h2 = no_room_double_booking(test_mini_schedule)
h3 = room_capacity_ok(test_mini_schedule, mini_exams, mini_rooms)
h4 = one_exam_per_student_per_day(test_mini_schedule, mini_cm, mini_timeslots)
s1 = exams_spread_evenly(test_mini_schedule, mini_timeslots)
s2 = no_dept_same_day(test_mini_schedule, mini_timeslots)
s3 = prefer_morning_slots(test_mini_schedule, mini_timeslots)
expected_score = -(( h1+h2+h3+h4)*100 + (s1+s2+s3))

assert str(expected_score) in report, (
    f"Expected fitness score {expected_score} to appear in report.\n"
    f"Report was:\n{report}"
)

print("✓ explain_violations — report generated successfully")
print()
print("═" * 60)
print("SAMPLE REPORT OUTPUT (mini test schedule):")
print("═" * 60)
print(report)

---
## Part 5 — End-to-End Test on Real Data

Loads all real data, constructs three schedules (bad, random, greedy), and runs the full
fitness pipeline on each to confirm everything works together.

In [ ]:
# ── End-to-end with real data ─────────────────────────────────────────────────
import random

exam_codes   = list(exams.keys())
room_ids     = [r['room_id'] for r in rooms]
n_slots      = len(timeslots)

# Schedule 1: worst case — all exams in slot 0, room 0
worst_schedule = [
    {'exam': e, 'room_id': room_ids[0], 'slot_id': 0}
    for e in exam_codes
]

# Schedule 2: random assignment
random.seed(42)
random_schedule = [
    {'exam': e,
     'room_id': random.choice(room_ids),
     'slot_id': random.randint(0, n_slots - 1)}
    for e in exam_codes
]

# Schedule 3: round-robin (one exam per slot, cycling through rooms)
rr_schedule = [
    {'exam': e,
     'room_id': room_ids[i % len(room_ids)],
     'slot_id': i % n_slots}
    for i, e in enumerate(exam_codes)
]

for name, sched in [('Worst', worst_schedule), ('Random', random_schedule), ('Round-robin', rr_schedule)]:
    h = count_hard_violations(sched, conflict_matrix, exams, rooms, timeslots)
    s = count_soft_violations(sched, timeslots)
    f = fitness(sched, conflict_matrix, exams, rooms, timeslots)
    assert f == -(h * 100 + s), f"{name}: fitness formula mismatch"
    assert f <= 0,               f"{name}: fitness must be <= 0"
    print(f"  {name:<14}  hard={h:>4}  soft={s:>4}  fitness={f}")

# Worst must be worse than round-robin
f_worst = fitness(worst_schedule, conflict_matrix, exams, rooms, timeslots)
f_rr    = fitness(rr_schedule,    conflict_matrix, exams, rooms, timeslots)
assert f_worst <= f_rr, "Worst-case schedule must score ≤ round-robin"

print()
print("✓ End-to-end real-data pipeline passed.")

---
## Summary

In [ ]:
print("═" * 55)
print("ALL TESTS PASSED")
print("═" * 55)
print()
print("Data loader functions tested:")
print("  ✓ load_exams")
print("  ✓ load_students")
print("  ✓ load_rooms")
print("  ✓ load_timeslots")
print("  ✓ build_conflict_matrix")
print("  ✓ load_conflict_matrix")
print()
print("Constraint functions tested:")
print("  ✓ no_student_clash          (hard)")
print("  ✓ no_room_double_booking    (hard)")
print("  ✓ room_capacity_ok          (hard)")
print("  ✓ one_exam_per_student_per_day (hard)")
print("  ✓ exams_spread_evenly       (soft)")
print("  ✓ no_dept_same_day          (soft)")
print("  ✓ prefer_morning_slots      (soft)")
print("  ✓ count_hard_violations     (aggregator)")
print("  ✓ count_soft_violations     (aggregator)")
print("  ✓ fitness                   (scorer)")
print("  ✓ explain_violations        (reporter)")
print()
print("  ✓ End-to-end real-data pipeline")